# Model 7 — Random Forest, Cases 1–3

**Idea:** grow many regression trees and average their forecasts.

- A **regression tree** splits the data with yes/no questions ("is yesterday's GHI ≤ 300?"), choosing at each step the
  question that most reduces the squared error. Each leaf predicts the mean power of the training rows that end there.
  Unlike a straight line, a tree can bend — e.g. follow the plant's midday plateau at ~16–17 MW.
- **Bagging:** each tree is trained on a bootstrap sample (rows drawn with replacement), and at each split only a random
  subset of inputs (`max_features`) is considered. The trees therefore differ, and averaging them reduces variance.

**Tuned on validation (lowest RMSE):** `max_depth`, `min_samples_leaf`, `max_features`. `n_estimators` is fixed at 300.

Same data, inputs, split, scoring window and metrics as the linear-model notebooks.

In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

## 1. Settings

In [ ]:
# Works whether the notebook is opened from src/ or from the project folder.
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
CLEAN_CSV = ROOT / "Data" / "Clean" / "Winter_Clean.csv"
RESULTS = ROOT / "results"

CAPACITY_MW = 21.5
BAND_MW = 0.07 * CAPACITY_MW  # ±7% of capacity (GERC 2019 tolerance) = 1.505 MW

# Scoring window 07:00-18:15 (blocks 29-74): the plant never generates outside it.
DAY_FIRST_BLOCK, DAY_LAST_BLOCK = 29, 74

# Chronological split by target day (dates inclusive).
SPLITS = {
    "train": ("2025-10-15", "2026-01-14"),
    "val": ("2026-01-15", "2026-02-07"),
    "test": ("2026-02-08", "2026-02-14"),
}

# Short names for the CSV columns.
RENAME = {
    "Act. Power (MW)": "power",
    "W-GHI (W/m2)": "ghi",
    "W-MODT- (°C)": "modt",
    "W-AMBT (°C)": "ambt",
    "W-WSPD (m/s)": "wspd",
    "W-WDIR- (Degree)": "wdir",
}
WEATHER = ["ghi", "modt", "ambt", "wspd", "wdir"]

CASES = {
    "Case 1": ["power_D"],
    "Case 2": ["power_D"] + [f"{w}_D" for w in WEATHER],
    "Case 3": ["power_D"] + [f"{w}_D" for w in WEATHER] + [f"{w}_D+1" for w in WEATHER],
}

N_ESTIMATORS = 300
SEED = 42  # fixes the bootstrap samples and feature choices, so results are reproducible

# Candidate settings. None = grow until leaves are pure / hit min_samples_leaf.
MAX_DEPTHS = [3, 5, 8, 12, None]
MIN_SAMPLES_LEAF = [1, 5, 20, 50]
MAX_FEATURES = [0.33, 0.67, 1.0]  # fraction of inputs tried at each split

## 2. Load the clean data and label each day with its split

In [ ]:
df = pd.read_csv(CLEAN_CSV, parse_dates=["TimeStamp", "Date"]).rename(columns=RENAME)

df["split"] = None
for name, (start, end) in SPLITS.items():
    df.loc[df["Date"].between(start, end), "split"] = name
assert df["split"].notna().all(), "some dates fall outside every split"

df.groupby("split")["Date"].agg(["min", "max", "nunique"]).loc[["train", "val", "test"]]

## 3. Build the inputs (identical to MLR)

`_D` = value at the same block on the previous day, `_D+1` = value on the target day (Case 3 only).

In [ ]:
by_block = df.groupby("Block")
for col in ["power"] + WEATHER:
    df[f"{col}_D"] = by_block[col].shift(1)
for w in WEATHER:
    df[f"{w}_D+1"] = df[w]

# 15 Oct has no previous day, so it has no inputs.
df = df.dropna(subset=["power_D"]).reset_index(drop=True)

daytime = df[df["Block"].between(DAY_FIRST_BLOCK, DAY_LAST_BLOCK)].copy()
train = daytime[daytime["split"] == "train"]
val = daytime[daytime["split"] == "val"]

print("daytime rows per split:", daytime["split"].value_counts()[["train", "val", "test"]].to_dict())

## 4. Metrics

In [ ]:
def evaluate(actual, forecast):
    err = forecast - actual
    mae = err.abs().mean()
    rmse = np.sqrt((err**2).mean())
    # R² = 1 - SS_res / SS_tot: share of variation explained vs always predicting the mean.
    r2 = 1 - (err**2).sum() / ((actual - actual.mean()) ** 2).sum()
    return {
        "MAE (MW)": mae,
        "RMSE (MW)": rmse,
        "nMAE (%)": 100 * mae / CAPACITY_MW,
        "nRMSE (%)": 100 * rmse / CAPACITY_MW,
        "Within ±7% (%)": 100 * (err.abs() <= BAND_MW).mean(),
        "R² (%)": 100 * r2,
        "Blocks": len(err),
    }


def skill_score(rmse, rmse_persistence):
    # 0 = no better than persistence, positive = better, negative = worse.
    return 100 * (1 - rmse / rmse_persistence)


def rf_model(max_depth, min_samples_leaf, max_features):
    return RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=SEED,
        n_jobs=-1,
    )

## 5. Tune on validation

Every combination of `max_depth` × `min_samples_leaf` × `max_features` is fitted on training and scored on validation.
Case 1 has a single input, so `max_features` makes no difference there and only 1.0 is tried.
The combination with the lowest validation RMSE is kept. Training RMSE is recorded too, to see overfitting.

In [ ]:
search = []
for case, features in CASES.items():
    feature_options = [1.0] if len(features) == 1 else MAX_FEATURES
    for depth, leaf, mf in product(MAX_DEPTHS, MIN_SAMPLES_LEAF, feature_options):
        model = rf_model(depth, leaf, mf).fit(train[features], train["power"])
        search.append({
            "case": case,
            "max_depth": depth,
            "min_samples_leaf": leaf,
            "max_features": mf,
            "train RMSE (MW)": evaluate(train["power"], model.predict(train[features]))["RMSE (MW)"],
            "val RMSE (MW)": evaluate(val["power"], np.clip(model.predict(val[features]), 0, CAPACITY_MW))["RMSE (MW)"],
        })
search = pd.DataFrame(search)

best = search.loc[search.groupby("case")["val RMSE (MW)"].idxmin()].set_index("case")
best

## 6. Depth vs error (under- and overfitting)

Keeping each case's chosen `min_samples_leaf` and `max_features`, vary only `max_depth` from 1 to 20 (and unlimited).
Shallow trees underfit (both errors high); deep trees fit training ever better while validation stops improving.

In [ ]:
DEPTH_SWEEP = list(range(1, 21)) + [None]

sweep = []
for case, features in CASES.items():
    leaf, mf = best.loc[case, "min_samples_leaf"], best.loc[case, "max_features"]
    for depth in DEPTH_SWEEP:
        model = rf_model(depth, leaf, mf).fit(train[features], train["power"])
        sweep.append({
            "case": case,
            "max_depth": depth,
            "train RMSE (MW)": evaluate(train["power"], model.predict(train[features]))["RMSE (MW)"],
            "val RMSE (MW)": evaluate(val["power"], np.clip(model.predict(val[features]), 0, CAPACITY_MW))["RMSE (MW)"],
        })
sweep = pd.DataFrame(sweep)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (case, part) in zip(axes, sweep.groupby("case")):
    x = part["max_depth"].fillna(22)  # plot "unlimited" at the far right
    ax.plot(x, part["train RMSE (MW)"], marker=".", label="Training")
    ax.plot(x, part["val RMSE (MW)"], marker=".", label="Validation")
    ax.set_xticks([1, 5, 10, 15, 20, 22], ["1", "5", "10", "15", "20", "None"])
    ax.set_xlabel("max_depth")
    ax.set_title(f"{case} (min_samples_leaf={best.loc[case, 'min_samples_leaf']}, max_features={best.loc[case, 'max_features']})", fontsize=9)
axes[0].set_ylabel("RMSE (MW)")
axes[0].legend()
plt.tight_layout()
plt.show()

## 7. Fit the final model per case with its chosen settings

In [ ]:
models = {}
for case, features in CASES.items():
    b = best.loc[case]
    max_depth = None if pd.isna(b["max_depth"]) else int(b["max_depth"])
    model = rf_model(max_depth, int(b["min_samples_leaf"]), b["max_features"]).fit(train[features], train["power"])
    daytime[case] = np.clip(model.predict(daytime[features]), 0, CAPACITY_MW)
    models[case] = model

## 8. Scores: every case on every split

In [ ]:
rows = []
for case in CASES:
    for split in ["train", "val", "test"]:
        part = daytime[daytime["split"] == split]
        scores = evaluate(part["power"], part[case])
        rmse_persistence = evaluate(part["power"], part["power_D"])["RMSE (MW)"]
        rows.append({
            "case": case,
            "split": split,
            **scores,
            "Skill vs persistence (%)": skill_score(scores["RMSE (MW)"], rmse_persistence),
        })

metrics = pd.DataFrame(rows).round(3)
metrics

Validation only, side by side (this is the table used to compare models):

In [ ]:
metrics[metrics["split"] == "val"].set_index("case").drop(columns=["split", "Blocks"])

## 9. Which inputs does the forest rely on?

**Permutation importance on validation:** shuffle one input's values and measure how much validation RMSE rises.
A big rise means the model depends on that input. (Computed on validation, not training, so it reflects what generalises.)

In [ ]:
importance = {}
for case, features in CASES.items():
    result = permutation_importance(
        models[case], val[features], val["power"],
        scoring="neg_root_mean_squared_error", n_repeats=5, random_state=SEED, n_jobs=-1,
    )
    importance[case] = pd.Series(result.importances_mean, index=features)
importance = pd.DataFrame(importance).reindex(CASES["Case 3"])
importance.round(3)  # rise in validation RMSE (MW) when the input is shuffled

## 10. Where does it fail? Worst validation days (Case 2, the realistic case)

In [ ]:
daily_mae = (
    daytime[daytime["split"] == "val"]
    .assign(abs_err=lambda x: (x["Case 2"] - x["power"]).abs())
    .groupby("Date")["abs_err"]
    .mean()
    .round(2)
)
daily_mae.sort_values(ascending=False).head(5)

In [ ]:
# Same three validation days as the other notebooks: clear, cloudy, clear.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, day in zip(axes, ["2026-01-25", "2026-01-26", "2026-01-27"]):
    d = daytime[daytime["Date"] == day]
    hours = (d["Block"] - 1) / 4
    ax.fill_between(hours, d["power"] - BAND_MW, d["power"] + BAND_MW, alpha=0.2, label="±7% band")
    ax.plot(hours, d["power"], color="black", label="Actual")
    ax.plot(hours, d["power_D"], ":", label="Persistence")
    for case in CASES:
        ax.plot(hours, d[case], "--", label=f"RF {case}")
    ax.set_title(day)
    ax.set_xlabel("Hour of day")
axes[0].set_ylabel("Power (MW)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 11. Save forecasts, scores, the search and the depth sweep

In [ ]:
(RESULTS / "predictions").mkdir(parents=True, exist_ok=True)
(RESULTS / "metrics").mkdir(parents=True, exist_ok=True)

# Every block, with night forecasts set to 0.
out = df[["TimeStamp", "Date", "Block", "split", "power"]].rename(columns={"power": "actual"})
for case in CASES:
    out[f"RF {case}"] = daytime[case].reindex(out.index).fillna(0.0)

out.to_csv(RESULTS / "predictions" / "random_forest.csv", index=False)
metrics.to_csv(RESULTS / "metrics" / "random_forest.csv", index=False)
search.to_csv(RESULTS / "metrics" / "random_forest_search.csv", index=False)
sweep.to_csv(RESULTS / "metrics" / "random_forest_depth_sweep.csv", index=False)
importance.to_csv(RESULTS / "metrics" / "random_forest_importance.csv")